# RDF Graph Builder from _graph.json

Build RDF graph from esgvoc _graph.json files using rdflib

In [ ]:
import json
import sys
from pathlib import Path
from collections import defaultdict

try:
    from rdflib import Graph, Namespace, URIRef, Literal, RDF, RDFS
    from rdflib.namespace import SKOS
except ImportError:
    import subprocess
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'rdflib'])
    from rdflib import Graph, Namespace, URIRef, Literal, RDF, RDFS
    from rdflib.namespace import SKOS

In [ ]:
BASE_DIR = Path('/Users/daniel.ellis/WIPwork/Essential-Model-Documentation')
graph_files = sorted(BASE_DIR.rglob('_graph.json'))
print(f'Found {len(graph_files)} files:')
for f in graph_files:
    print(f'  - {f.relative_to(BASE_DIR)}')

In [ ]:
def load_graph_json(file_path):
    try:
        with open(file_path, 'r') as f:
            return json.load(f)
    except Exception as e:
        print(f'Error: {e}')
        return {}

graph_data = {}
for gf in graph_files:
    folder_name = gf.parent.name
    data = load_graph_json(gf)
    graph_data[folder_name] = data
    print(f'Loaded {folder_name}')

In [ ]:
def extract_uris(obj):
    uris = set()
    def recurse(item):
        if isinstance(item, dict):
            if '@id' in item:
                uris.add(item['@id'])
            if '@type' in item:
                t = item['@type']
                if isinstance(t, str):
                    uris.add(t)
                elif isinstance(t, list):
                    uris.update(t)
            for v in item.values():
                recurse(v)
        elif isinstance(item, list):
            for i in item:
                recurse(i)
    recurse(obj)
    return uris

all_uris = set()
for folder_name, data in graph_data.items():
    uris = extract_uris(data)
    all_uris.update(uris)
    print(f'{folder_name}: {len(uris)} URIs')

print(f'\nTotal unique URIs: {len(all_uris)}')

In [ ]:
g = Graph()
ESGVOC = Namespace('http://example.org/esgvoc/')
g.bind('esgvoc', ESGVOC)
g.bind('skos', SKOS)

def to_uri(uri_str):
    if uri_str.startswith('http://') or uri_str.startswith('https://'):
        return URIRef(uri_str)
    return ESGVOC[uri_str]

triples = 0
for folder_name, data in graph_data.items():
    folder_uri = ESGVOC[folder_name]
    g.add((folder_uri, RDF.type, SKOS.Collection))
    g.add((folder_uri, RDFS.label, Literal(folder_name)))
    triples += 2
    
    if 'contents' in data and isinstance(data['contents'], list):
        for item in data['contents']:
            if isinstance(item, dict) and '@id' in item:
                item_uri = to_uri(item['@id'])
                g.add((folder_uri, SKOS.member, item_uri))
                triples += 1
                
                if '@type' in item:
                    types = item['@type'] if isinstance(item['@type'], list) else [item['@type']]
                    for t in types:
                        g.add((item_uri, RDF.type, to_uri(t)))
                        triples += 1

print(f'Added {triples} triples')
print(f'Graph size: {len(g)} triples')

In [ ]:
print('='*60)
print('GRAPH STATISTICS')
print('='*60)
print(f'Total triples: {len(g)}')
print(f'Unique subjects: {len(set(s for s, _, _ in g))}')
print(f'Unique predicates: {len(set(p for _, p, _ in g))}')
print(f'Unique objects: {len(set(o for _, _, o in g))}')

pred_counts = defaultdict(int)
for _, p, _ in g:
    pred_counts[str(p)] += 1

print('\nTriples by predicate:')
for pred, count in sorted(pred_counts.items(), key=lambda x: -x[1]):
    print(f'  {pred}: {count}')

In [ ]:
output_dir = BASE_DIR / 'rdf_output'
output_dir.mkdir(exist_ok=True)

formats = [('turtle', 'ttl'), ('xml', 'rdf'), ('json-ld', 'jsonld'), ('nt', 'nt')]
for fmt, ext in formats:
    output_file = output_dir / f'combined_graph.{ext}'
    g.serialize(destination=str(output_file), format=fmt)
    print(f'Exported to {output_file}')

print(f'\nAll files saved to: {output_dir}')

In [ ]:
query = """
PREFIX skos: <http://www.w3.org/2004/02/skos/core#>
SELECT ?collection ?member
WHERE {
    ?collection a skos:Collection .
    ?collection skos:member ?member .
}
"""

print('Collections and members:')
results = g.query(query)
for row in results:
    print(f'{row.collection} -> {row.member}')